# Italian Food Inflation Nowcast — Esselunga

Notebook unico per costruire un proxy ad alta frequenza dei prezzi alimentari Esselunga, CAP **20141**.

Il risultato è un **retailer-specific online-price proxy**, non un indice ufficiale Istat/IPCA.

## 1. Installazione (solo la prima volta)

In [ ]:
# Run this cell only once.
%pip install pandas numpy matplotlib playwright openpyxl
!playwright install chromium

## 2. Import e configurazione

In [ ]:
from __future__ import annotations
import hashlib, json, re, sqlite3
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError

CAP = "20141"
HEADLESS = False  # Set True only after the first successful run
BASE_DIR = Path.cwd() / "esselunga_nowcast_data"
RAW_DIR = BASE_DIR / "raw"
DB_PATH = BASE_DIR / "prezzi_esselunga.sqlite3"
CSV_PATH = BASE_DIR / "ultimo_snapshot.csv"
PROFILE_DIR = BASE_DIR / "browser_profile"
for folder in (BASE_DIR, RAW_DIR, PROFILE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Cartella dati:", BASE_DIR)

## 3. Categorie ECOICOP e pesi relativi

In [ ]:
# Italian labels highlight local-market knowledge; code comments remain in English.
# These starting weights are indicative. Replace them with the exact official Italian ECOICOP weights.
# The selected weights are normalized automatically to sum to 100%.
CATEGORIE = {
    "Pane e cereali": {"ecoicop":"01.1.1", "peso_grezzo":16.0, "url":"https://spesaonline.esselunga.it/commerce/nav/supermercato/store/menu/300000001002050/pane-e-sostitutivi"},
    "Carne": {"ecoicop":"01.1.2", "peso_grezzo":24.0, "url":"https://spesaonline.esselunga.it/commerce/nav/supermercato/store/menu/300000001002007/carne"},
    "Pesce e prodotti ittici": {"ecoicop":"01.1.3", "peso_grezzo":8.0, "url":"https://spesaonline.esselunga.it/commerce/nav/supermercato/store/menu/600000001048549/esselunga-sushi"},
    "Latte, formaggi e uova": {"ecoicop":"01.1.4", "peso_grezzo":18.0, "url":"https://spesaonline.esselunga.it/commerce/nav/supermercato/store/menu/600000001047248/latte-yogurt-e-uova"},
    "Oli e grassi": {"ecoicop":"01.1.5", "peso_grezzo":5.0, "url":"https://spesaonline.esselunga.it/commerce/nav/supermercato/store/menu/300000001002513/olio-extra-vergine"},
    "Frutta": {"ecoicop":"01.1.6", "peso_grezzo":10.0, "url":"https://spesaonline.esselunga.it/commerce/nav/supermercato/store/menu/600000001047175/frutta"},
    "Verdura, tuberi e legumi": {"ecoicop":"01.1.7", "peso_grezzo":13.0, "url":"https://spesaonline.esselunga.it/commerce/nav/supermercato/store/menu/600000001047177/verdura"},
    "Zucchero e dolciumi": {"ecoicop":"01.1.8", "peso_grezzo":6.0, "url":"https://spesaonline.esselunga.it/commerce/nav/supermercato/store/menu/300000001019692/patatine-e-dolciumi"},
}
totale = sum(v["peso_grezzo"] for v in CATEGORIE.values())
for v in CATEGORIE.values(): v["peso_relativo"] = v["peso_grezzo"] / totale
pesi_df = pd.DataFrame([{"categoria":k,"ECOICOP":v["ecoicop"],"peso_relativo_%":100*v["peso_relativo"]} for k,v in CATEGORIE.items()])
pesi_df

## 4. Parser e database

In [ ]:
PRICE_KEYS=("price","sellingPrice","currentPrice","finalPrice","effectivePrice","unitPrice","offerPrice","discountedPrice","value")
NAME_KEYS=("name","description","title","productName","displayName")
ID_KEYS=("id","productId","sku","code","ean","gtin")
BRAND_KEYS=("brand","brandName","manufacturer")

def clean_text(value: Any):
    if value is None: return None
    value=re.sub(r"\s+"," ",str(value)).strip()
    return value or None

def parse_price(value: Any):
    # Convert common Italian and JSON price formats into a positive float.
    if value is None or isinstance(value,bool): return None
    if isinstance(value,(int,float)):
        x=float(value)
        if x>1000 and x.is_integer(): x/=100
        return x if 0.01<=x<=10000 else None
    text=str(value).replace("€","").replace("EUR","").strip()
    text=re.sub(r"[^0-9,.-]","",text)
    if not text: return None
    text=text.replace(".","").replace(",",".") if "," in text and "." in text else text.replace(",",".")
    try: x=float(text)
    except ValueError: return None
    return x if 0.01<=x<=10000 else None

def first_value(obj:dict, keys:Iterable[str]):
    for key in keys:
        if key in obj and obj[key] not in (None,"",[],{}): return obj[key]
    return None

def walk_json(node:Any):
    # Recursively yield every dictionary contained in a JSON response.
    if isinstance(node,dict):
        yield node
        for value in node.values(): yield from walk_json(value)
    elif isinstance(node,list):
        for value in node: yield from walk_json(value)

def stable_product_id(category,name,brand,raw_id):
    # Prefer the retailer identifier; otherwise create a stable hash.
    if raw_id not in (None,""): return str(raw_id)
    token=f"{category}|{brand or ''}|{name}".lower().encode("utf-8")
    return hashlib.sha1(token).hexdigest()[:20]

def extract_products(payload,categoria,source_url):
    products=[]; seen=set()
    for obj in walk_json(payload):
        name=clean_text(first_value(obj,NAME_KEYS))
        if not name or len(name)<3: continue
        price=None
        for key in PRICE_KEYS:
            if key in obj:
                candidate=obj[key]
                if isinstance(candidate,dict): candidate=first_value(candidate,("value","amount","price"))
                price=parse_price(candidate)
                if price is not None: break
        if price is None: continue
        brand_raw=first_value(obj,BRAND_KEYS)
        if isinstance(brand_raw,dict): brand_raw=first_value(brand_raw,NAME_KEYS)
        brand=clean_text(brand_raw)
        pid=stable_product_id(categoria,name,brand,first_value(obj,ID_KEYS))
        key=(categoria,pid,round(price,4))
        if key in seen: continue
        seen.add(key)
        regular=parse_price(first_value(obj,("regularPrice","originalPrice","listPrice","oldPrice")))
        products.append({"product_id":pid,"categoria":categoria,"nome_prodotto":name,"brand":brand,
                         "prezzo_effettivo":price,"prezzo_regolare":regular,
                         "promozione":int(bool(regular and regular>price)),"source_url":source_url})
    return products

def init_database():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("CREATE TABLE IF NOT EXISTS observations (run_id TEXT, collected_at TEXT, cap TEXT, categoria TEXT, ecoicop TEXT, product_id TEXT, nome_prodotto TEXT, brand TEXT, prezzo_effettivo REAL, prezzo_regolare REAL, promozione INTEGER, source_url TEXT, PRIMARY KEY(run_id,categoria,product_id))")
        conn.execute("CREATE TABLE IF NOT EXISTS runs (run_id TEXT PRIMARY KEY, collected_at TEXT, is_baseline INTEGER, n_products INTEGER)")
        conn.commit()

def save_snapshot(df,run_id,collected_at):
    init_database()
    with sqlite3.connect(DB_PATH) as conn:
        is_baseline=int(conn.execute("SELECT COUNT(*) FROM runs").fetchone()[0]==0)
        rows=df.copy(); rows["run_id"]=run_id; rows["collected_at"]=collected_at; rows["cap"]=CAP
        rows["ecoicop"]=rows["categoria"].map({k:v["ecoicop"] for k,v in CATEGORIE.items()})
        cols=["run_id","collected_at","cap","categoria","ecoicop","product_id","nome_prodotto","brand","prezzo_effettivo","prezzo_regolare","promozione","source_url"]
        rows[cols].to_sql("observations",conn,if_exists="append",index=False)
        conn.execute("INSERT INTO runs VALUES (?,?,?,?)",(run_id,collected_at,is_baseline,len(rows)))
        conn.commit()

def load_runs():
    init_database()
    with sqlite3.connect(DB_PATH) as conn: return pd.read_sql_query("SELECT * FROM runs ORDER BY collected_at",conn)

def load_observations():
    init_database()
    with sqlite3.connect(DB_PATH) as conn: return pd.read_sql_query("SELECT * FROM observations",conn)

## 5. Raccolta prezzi — esegui questa cella a ogni nuova rilevazione

In [ ]:
def scrape_esselunga(headless=HEADLESS):
    collected_at=datetime.now(timezone.utc).isoformat(timespec="seconds")
    run_id=datetime.now().strftime("%Y%m%d_%H%M%S")
    all_products=[]
    with sync_playwright() as p:
        context=p.chromium.launch_persistent_context(user_data_dir=str(PROFILE_DIR),headless=headless,viewport={"width":1440,"height":1000},locale="it-IT")
        page=context.pages[0] if context.pages else context.new_page()
        captured=[]
        def handle_response(response):
            # Capture JSON because the catalogue is loaded dynamically.
            if "json" not in response.headers.get("content-type","").lower(): return
            try: captured.append((response.url,response.json()))
            except Exception: pass
        page.on("response",handle_response)
        page.goto(next(iter(CATEGORIE.values()))["url"],wait_until="domcontentloaded",timeout=120000)
        if not headless:
            print("Nel browser: accetta i cookie, imposta CAP 20141 e seleziona consegna/negozio.")
            input("Quando vedi prodotti e prezzi, torna qui e premi INVIO: ")
        for categoria,info in CATEGORIE.items():
            captured.clear(); print("Raccolta:",categoria)
            try:
                page.goto(info["url"],wait_until="domcontentloaded",timeout=120000)
                page.wait_for_timeout(6000)
                for _ in range(8): page.mouse.wheel(0,1600); page.wait_for_timeout(800)
            except PlaywrightTimeoutError: print("Timeout; uso i dati già caricati.")
            category_products=[]
            raw_dir=RAW_DIR/run_id/re.sub(r"[^a-z0-9]+","_",categoria.lower()).strip("_")
            raw_dir.mkdir(parents=True,exist_ok=True)
            for idx,(response_url,payload) in enumerate(captured):
                try: (raw_dir/f"response_{idx:04d}.json").write_text(json.dumps(payload,ensure_ascii=False),encoding="utf-8")
                except Exception: pass
                category_products.extend(extract_products(payload,categoria,response_url))
            if category_products:
                temp=pd.DataFrame(category_products).sort_values(["product_id","prezzo_effettivo"]).drop_duplicates("product_id")
                category_products=temp.to_dict("records")
            print(" prodotti identificati:",len(category_products))
            all_products.extend(category_products)
        context.close()
    if not all_products: raise RuntimeError("Nessun prodotto identificato. Ripeti con HEADLESS=False e verifica che il catalogo sia visibile.")
    df=pd.DataFrame(all_products); df["run_id"]=run_id; df["collected_at"]=collected_at
    df.to_csv(CSV_PATH,index=False,encoding="utf-8-sig"); save_snapshot(df,run_id,collected_at)
    print("Snapshot salvato:",len(df),"prodotti")
    return df

snapshot=scrape_esselunga()
snapshot.groupby("categoria").size().rename("prodotti").to_frame()

## 6. Indici matched-product e aggregazione ponderata

In [ ]:
def geometric_mean(s):
    x=pd.to_numeric(s,errors="coerce").dropna(); x=x[x>0]
    return float(np.exp(np.log(x).mean())) if len(x) else np.nan

def calculate_indices():
    runs=load_runs(); obs=load_observations()
    baseline_run=runs.loc[runs["is_baseline"]==1,"run_id"].iloc[0]
    baseline=obs[obs["run_id"]==baseline_run].rename(columns={"prezzo_effettivo":"prezzo_baseline"})
    category_rows=[]; aggregate_rows=[]
    for _,run in runs.iterrows():
        current=obs[obs["run_id"]==run["run_id"]]
        matched=current.merge(baseline[["categoria","product_id","prezzo_baseline"]],on=["categoria","product_id"])
        matched["price_relative"]=matched["prezzo_effettivo"]/matched["prezzo_baseline"]
        matched=matched[matched["price_relative"].between(.25,4.0)]
        cat_indices={}
        for categoria,info in CATEGORIE.items():
            block=matched[matched["categoria"]==categoria].copy()
            base_n=baseline[baseline["categoria"]==categoria]["product_id"].nunique(); n=block["product_id"].nunique()
            if len(block):
                lo,hi=block["price_relative"].quantile([.02,.98]); rel=block["price_relative"].clip(lo,hi)
                index=100*geometric_mean(rel); up=100*(block["price_relative"]>1.001).mean(); down=100*(block["price_relative"]<.999).mean()
            else: index=up=down=np.nan
            cat_indices[categoria]=index
            category_rows.append({"run_id":run["run_id"],"data":run["collected_at"],"categoria":categoria,"ECOICOP":info["ecoicop"],
                                  "peso_relativo":info["peso_relativo"],"indice":index,"variazione_dalla_baseline_%":index-100 if pd.notna(index) else np.nan,
                                  "prodotti_matched":n,"coverage_%":100*n/base_n if base_n else np.nan,"quota_aumenti_%":up,"quota_riduzioni_%":down})
        available={k:v for k,v in cat_indices.items() if pd.notna(v)}; covered=sum(CATEGORIE[k]["peso_relativo"] for k in available)
        agg=sum(available[k]*CATEGORIE[k]["peso_relativo"]/covered for k in available) if covered else np.nan
        aggregate_rows.append({"run_id":run["run_id"],"data":run["collected_at"],"indice_proxy":agg,"variazione_dalla_baseline_%":agg-100 if pd.notna(agg) else np.nan,"peso_coperto_%":100*covered})
    return pd.DataFrame(category_rows),pd.DataFrame(aggregate_rows)

indici_categoria,indice_aggregato=calculate_indices()
indice_aggregato

## 7. Risultato e grafici

In [ ]:
latest_run=indice_aggregato.sort_values("data").iloc[-1]["run_id"]
latest_categories=indici_categoria[indici_categoria["run_id"]==latest_run].copy()
latest_aggregate=indice_aggregato[indice_aggregato["run_id"]==latest_run].iloc[0]
print("ESSELUNGA MILANO FOOD PRICE PROXY")
print("Variazione dalla baseline: %.2f%%" % latest_aggregate["variazione_dalla_baseline_%"])
display(latest_categories.sort_values("peso_relativo",ascending=False))

history=indice_aggregato.copy(); history["data"]=pd.to_datetime(history["data"]); history=history.sort_values("data")
plt.figure(figsize=(10,5)); plt.plot(history["data"],history["indice_proxy"],marker="o"); plt.axhline(100,linewidth=1)
plt.title("Esselunga Milano Food Price Proxy — baseline = 100"); plt.xlabel("Data"); plt.ylabel("Indice"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

chart=latest_categories.sort_values("variazione_dalla_baseline_%")
plt.figure(figsize=(10,6)); plt.barh(chart["categoria"],chart["variazione_dalla_baseline_%"]); plt.axvline(0,linewidth=1)
plt.title("Variazione dalla baseline per categoria"); plt.xlabel("Variazione %"); plt.tight_layout(); plt.show()

## 8. Esportazione

In [ ]:
REPORT_XLSX=BASE_DIR/"report_inflazione_esselunga.xlsx"
REPORT_HTML=BASE_DIR/"report_inflazione_esselunga.html"
with pd.ExcelWriter(REPORT_XLSX,engine="openpyxl") as writer:
    indice_aggregato.to_excel(writer,sheet_name="Indice aggregato",index=False)
    indici_categoria.to_excel(writer,sheet_name="Categorie",index=False)
    pesi_df.to_excel(writer,sheet_name="Pesi",index=False)
html=("<html><head><meta charset='utf-8'><title>Italian Food Inflation Nowcast</title></head><body>"
      "<h1>Italian Food Inflation Nowcast</h1><p>Esselunga CAP "+CAP+". Proxy retailer-specifico, non indice ufficiale Istat.</p>"
      "<h2>Variazione dalla baseline: %.2f%%</h2>" % latest_aggregate["variazione_dalla_baseline_%"]
      +latest_categories.to_html(index=False,float_format=lambda x:f"{x:.2f}")+"</body></html>")
REPORT_HTML.write_text(html,encoding="utf-8")
print("Excel:",REPORT_XLSX); print("HTML:",REPORT_HTML)

## Come usarlo

1. Esegui le celle dall'alto verso il basso.
2. Alla prima raccolta lascia `HEADLESS = False` e configura CAP/negozio nel browser.
3. La prima raccolta diventa automaticamente la baseline = 100.
4. Nei giorni successivi riapri il notebook ed esegui nuovamente dalla sezione 5 in poi.
5. Controlla sempre `coverage_%` e valida un campione di prezzi manualmente.

**Nota:** i pesi inseriti sono una struttura iniziale indicativa; prima di presentare il risultato come ponderato con pesi ufficiali, sostituiscili con gli esatti pesi italiani ECOICOP dell'anno di riferimento.